In [0]:
from pyspark.sql.functions import *

# Exploratory Data Analysis silver

In [0]:
# Läser in data från bronze

df = spark.table("marathos.bronze.results_raw")

display(df)

In [0]:
df_clean = (
    df
    .withColumnRenamed("Year of event", "event_year")
    .withColumnRenamed("Event dates", "event_dates")
    .withColumnRenamed("Event name", "event_name")
    .withColumnRenamed("Event distance/length", "event_distance_length")
    .withColumnRenamed("Event number of finishers", "event_number_of_finishers")
    .withColumnRenamed("Athlete performance", "athlete_performance")
    .withColumnRenamed("Athlete club", "athlete_club")
    .withColumnRenamed("Athlete country", "athlete_country")
    .withColumnRenamed("Athlete year of birth", "athlete_year_of_birth")
    .withColumnRenamed("Athlete gender", "athlete_gender")
    .withColumnRenamed("Athlete age category", "athlete_age_category")
    .withColumnRenamed("Athlete average speed", "athlete_average_speed")
    .withColumnRenamed("Athlete ID", "athlete_id")
)

display(df_clean)

In [0]:
df_clean.printSchema()

In [0]:
display(
    df_clean.select("event_distance_length")
    .distinct()
    .orderBy("event_distance_length")
)

In [0]:
display(
    df_clean.select("athlete_performance")
    .distinct()
)

In [0]:
display(
    df_clean.filter(
        col("athlete_performance").contains("d")
    )
)

In [0]:
df_clean = df_clean.filter(
    ~col("athlete_performance").contains("d")
)

display(df_clean)

In [0]:
display(
    df_clean.filter(
        col("athlete_performance").contains("d")
    )
)

In [0]:
df_clean = df_clean.withColumn(
    "event_type",
    when(col("event_distance_length").contains("km"), "distance")
    .when(col("event_distance_length").contains("mi"), "distance")
    .when(col("event_distance_length").contains("h"), "time")
    .otherwise("unknown")
)

display(
    df_clean.select(
        "event_distance_length",
        "event_type"
    ).distinct()
)

In [0]:
df_clean = df_clean.filter(
    ~col("event_distance_length").contains("Etappen")
)

display(df_clean)

In [0]:
display(
    df_clean.filter(
        col("event_distance_length").contains("Etappen")
    )
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

event_window = Window.orderBy("event_name")

df_clean = df_clean.withColumn(
    "event_id",
    dense_rank().over(event_window)
)

display(
    df_clean.select(
        "event_name",
        "event_id"
    ).distinct()
)

In [0]:
display(
    df_clean.filter(
        col("event_name").contains("Selva Costera")
    ).select(
        "event_name",
        "event_id"
    )
)

In [0]:
result_window = Window.orderBy(
    "event_name",
    "athlete_id",
    "athlete_performance"
)

df_clean = df_clean.withColumn(
    "result_id",
    dense_rank().over(result_window)
)

display(
    df_clean.select(
        "result_id",
        "event_name",
        "athlete_id",
        "athlete_performance"
    )
)

In [0]:
display(
    df_clean.filter(
        (col("event_type") == "distance") &
        (~col("athlete_performance").contains("h"))
    )
)

In [0]:
df_clean = df_clean.filter(
    ~(
        (col("event_type") == "distance") &
        (~col("athlete_performance").contains("h"))
    )
)

df_clean = df_clean.withColumn(
    "athlete_club",
    when(col("athlete_club").isNull(), "Unknown")
    .otherwise(trim(col("athlete_club")))
)

display(df_clean)

In [0]:
df_clean = df_clean.withColumn(
    "athlete_club",
    regexp_replace(col("athlete_club"), r"\*", "")
)

In [0]:
display(
    df_clean.select("athlete_club").distinct()
)

In [0]:
df_clean = df_clean.withColumn(
    "athlete_average_speed",
    expr("try_cast(athlete_average_speed as double)")
)

df_clean = df_clean.withColumn(
    "athlete_average_speed",
    round(col("athlete_average_speed"), 2)
)

display(df_clean)

In [0]:
df_clean = df_clean.withColumn(
    "event_date_clean",
    regexp_extract(col("event_dates"), r"(\d{2}\.\d{2}\.\d{4})$", 1)
)

df_clean = df_clean.withColumn(
    "event_date",
    to_date(col("event_date_clean"), "dd.MM.yyyy")
)

display(
    df_clean.select(
        "event_dates",
        "event_date_clean",
        "event_date"
    )
)

In [0]:
display(spark.table("marathos.silver.marathos_obt"))

In [0]:
spark.table("marathos.silver.marathos_obt").printSchema()